**`validate_delivery`**

Scores a delivered inventory against its out-of-band references and writes
the result into the delivery itself.

Two references, neither of them an input to the inventory:

- CHEER hand-labeled survey points, for per-class occupancy accuracy.
- Shovels building permits, for county-level agreement and confusion.

Everything lands in an `accuracies/` folder beside the bundle's four
files, as tables plus figures.

Only aggregate numbers are written there. The row-level linkage carries
survey addresses, so it stays in the cache.

# Configure

In [ ]:
import argparse
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import openplaces as op
from openplaces.io.delivery import delivery_accuracy_dir

# CHEER-specific configuration (counties, survey path, permit directory,
# band collapse) lives in cheer_linkage.py beside this notebook. Resolve
# the repository root from any working directory, so this works whether
# Jupyter runs from the notebook's folder or a script runs from the root.
root = Path.cwd()
while not (root / 'src' / 'openplaces').exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / 'notebooks' / '05_curate'))
from cheer_linkage import (  # noqa: E402
    CLASSES,
    COLLAPSE,
    COUNTIES,
    PERMIT_STRONG_TIERS,
    RECIPE_ID,
    link_ground_truth,
    load_permits,
    permit_counties,
    permit_tier,
    score_sources,
)

In [ ]:
parser = argparse.ArgumentParser(
    description='Score a delivered inventory and write the result into the delivery.'
)
parser.add_argument('--recipe_id', default=RECIPE_ID)
parser.add_argument('--admin_ids', nargs='*', default=None)
# Default None resolves to the recipe's own delivery folder, so the
# accuracies travel with the bundle they describe.
parser.add_argument('--out_dir', default=None)
parser.add_argument(
    '--min_scored',
    type=int,
    default=30,
    help='Counties with fewer scored rows are tabulated but left off the figure',
)
parser.add_argument(
    '--min_confusion_rows',
    type=int,
    default=300,
    help='Counties with fewer strong-tier permit rows are left out of the confusion',
)
parser.add_argument('--verbose', action='store_true')

# Test arguments

In [ ]:
ARGS_TEST = '--recipe_id US_footprint-cheer-2026 --verbose '

args_list = [x for x in ARGS_TEST.split(' ') if x != '']

args = parser.parse_args(args_list)

args

# Validate the delivery

## Where the accuracies go

In [ ]:
out_dir = Path(args.out_dir) if args.out_dir else delivery_accuracy_dir(args.recipe_id)
out_dir.mkdir(parents=True, exist_ok=True)

written = []


def publish(obj, name, **kwargs):
    """Write one table or figure into the delivery, and record it."""
    if isinstance(obj, mpl.figure.Figure):
        path = out_dir / f'{name}.png'
        obj.savefig(path, dpi=200, bbox_inches='tight')
    else:
        if obj is None or not len(obj):
            return None
        path = out_dir / f'{name}.csv'
        obj.to_csv(path, **kwargs)
    written.append(path.name)
    return path


print(f'accuracies -> {out_dir}')

## Chart style

One accent for the delivered vote, one neutral for the inputs it
arbitrates, and a single-hue ramp for magnitude.

The accent pair was checked with a colorblind-separation validator rather
than by eye: worst adjacent pair dE 20.3 under protanopia, 19.1 under
tritanopia, well clear of the 8.0 floor.

Every bar carries its value as text, so the figures survive being printed
in grayscale and never rest on color alone.

In [ ]:
# Accent = the delivered vote; neutral = the evidence it arbitrates.
# Deliberately not one hue per source: the question these figures answer
# is "how does the product compare with its inputs", which is a
# two-group contrast, not six identities.
VOTE_COLOR = '#9070C8'
INPUT_COLOR = '#B8B4C4'
GRID_COLOR = '#DDDBE3'
INK = '#2A2A32'
# Single hue, light to dark: the heatmap encodes magnitude, not identity.
MAGNITUDE_CMAP = 'Purples'

plt.rcParams.update(
    {
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.edgecolor': GRID_COLOR,
        'axes.labelcolor': INK,
        'text.color': INK,
        'xtick.color': INK,
        'ytick.color': INK,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'font.size': 9,
    }
)


def _bar_labels(ax, bars, fmt='{:.3f}', pad=0.006):
    """Label each bar at its end, so the figure reads without color."""
    for bar in bars:
        width = bar.get_width()
        if not np.isfinite(width):
            continue
        ax.text(
            width + pad,
            bar.get_y() + bar.get_height() / 2,
            fmt.format(width),
            va='center',
            ha='left',
            fontsize=8,
        )

## Occupancy against the CHEER hand labels

Per-class F1 for the delivered vote and for every input it arbitrates.

An input that outscores the vote on a class is evidence the vote is
discarding, which a single headline number cannot show.

In [ ]:
counties = tuple(args.admin_ids) if args.admin_ids else COUNTIES
linked = link_ground_truth(counties, verbose=args.verbose)
scores = score_sources(linked)
publish(scores, 'occupancy-accuracy-by-source', index=False)
print(f'{len(linked):,} linked survey points')
scores

In [ ]:
# One panel per class rather than grouped bars: the comparison the reader
# makes is within a class, across sources, and grouping would interleave
# the two.
plot_classes = [*CLASSES, 'ALL']
fig, axes = plt.subplots(
    1, len(plot_classes), figsize=(3.3 * len(plot_classes), 3.6), sharex=True
)
for ax, cls in zip(axes, plot_classes, strict=True):
    block = scores[scores['class'].eq(cls)].dropna(subset=['f1']).sort_values('f1')
    colors = [VOTE_COLOR if s == 'final_vote' else INPUT_COLOR for s in block['source']]
    bars = ax.barh(block['source'], block['f1'], color=colors, height=0.62)
    _bar_labels(ax, bars)
    ax.set_title(cls, fontsize=10, pad=8)
    ax.set_xlim(0, 1.15)
    ax.xaxis.grid(True, color=GRID_COLOR, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
axes[0].set_xlabel('F1 vs hand labels')
fig.suptitle(
    'Occupancy accuracy: the delivered vote (purple) against its inputs',
    y=1.04,
    fontsize=11,
)
publish(fig, 'occupancy-accuracy-by-source')
plt.show()

## Occupancy against building permits

Permits are the second out-of-band reference. Coverage varies enormously
by county, so agreement is always reported next to the count it rests on.

Only the strong tiers are scored: permits reaching a footprint by parcel
id, or by address with a unanimous mode over at least two permits.

In [ ]:
rows = []
confusions = []
for county in permit_counties():
    permits = load_permits(county)
    footprints = op.get_entities(args.recipe_id, county, missing='ignore', geom=False)
    if permits is None or footprints is None or footprints.empty:
        continue
    joined = footprints[['occupancy_type', 'priority_on_parcel']].join(
        permits, how='inner'
    )
    joined['vote'] = joined['occupancy_type'].astype(object).replace(COLLAPSE)
    joined['tier'] = permit_tier(joined)
    # Permit evidence is parcel-level, so a shed inherits the house's
    # class; only primary structures can be scored against it.
    primary = joined[
        joined['priority_on_parcel'].astype(object).isin(['primary', 'unknown'])
    ]
    strong = primary[
        primary['tier'].isin(PERMIT_STRONG_TIERS) & primary['vote'].notna()
    ]
    if strong.empty:
        continue
    rows.append(
        {
            'county': county,
            'n_scored': len(strong),
            'agreement': round(
                strong['vote'].eq(strong['occupancy_type_mode']).mean(), 4
            ),
            'coverage': round(primary['tier'].ne('none').mean(), 4),
        }
    )
    if len(strong) >= args.min_confusion_rows:
        confusions.append(strong.assign(county=county))

permit_accuracy = pd.DataFrame(rows).sort_values('agreement', ascending=False)
publish(permit_accuracy, 'permit-accuracy-by-county', index=False)
permit_accuracy

In [ ]:
# Sorting the full list by agreement puts counties with one or two scored
# rows on top at a flat 1.000, which reads as the best county in the
# delivery and is nothing of the kind. The table keeps every county; the
# figure shows only those with enough rows to rank, and carries n in the
# label so the reader never sees a rate without its denominator.
panel = permit_accuracy[permit_accuracy['n_scored'] >= args.min_scored].sort_values(
    'agreement'
)
thin = len(permit_accuracy) - len(panel)

y = np.arange(len(panel))
fig, ax = plt.subplots(figsize=(8.0, max(3.0, 0.42 * len(panel))))
ax.barh(
    y + 0.19,
    panel['agreement'],
    height=0.34,
    color=VOTE_COLOR,
    label='agreement with permit mode',
)
ax.barh(
    y - 0.19,
    panel['coverage'],
    height=0.34,
    color=INPUT_COLOR,
    label='share of footprints with permit evidence',
)
for i, row in enumerate(panel.itertuples()):
    ax.text(
        row.agreement + 0.008, i + 0.19, f'{row.agreement:.3f}', va='center', fontsize=8
    )
    ax.text(
        row.coverage + 0.008,
        i - 0.19,
        f'{row.coverage:.1%}',
        va='center',
        fontsize=8,
        color='#5A5A66',
    )
ax.set_yticks(
    y,
    [
        f'{c}  (n={n:,})'
        for c, n in zip(panel['county'], panel['n_scored'], strict=True)
    ],
)
ax.set_xlim(0, 1.15)
ax.set_xlabel('share')
ax.xaxis.grid(True, color=GRID_COLOR, linewidth=0.6)
ax.set_axisbelow(True)
ax.tick_params(length=0)
# On the figure, not the axes: the panel grows with the county
# count, so an axes-fraction offset that clears the x-label at 11
# counties collides with it at 30. bbox_inches='tight' keeps it in
# the saved image.
fig.legend(
    *ax.get_legend_handles_labels(),
    frameon=False,
    fontsize=8,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.0),
    ncol=2,
)
ax.set_title(
    'Permit agreement, and the coverage it rests on\n'
    f'{len(panel)} counties with at least {args.min_scored} scored rows'
    + (f'; {thin} thinner counties are in the table only' if thin else ''),
    fontsize=11,
    pad=10,
)
publish(fig, 'permit-accuracy-by-county')
plt.show()

## Where the vote and the permits disagree

Rows are the delivered class, columns are what the permits say, shaded by
row share.

The two vocabularies are not the same width, so a strong off-diagonal
block is not automatically an error. The delivery resolves occupancy far
more finely than a permit record does: `Church` landing in the permits'
`Institutional`, or `Wholesale` in `Industrial`, is agreement expressed
in coarser words.

What is worth reading is disagreement *within* a shared class -- a
delivered `Single-Family` the permits call `Multi-Family`, or the
`Manufactured Home` / `Single-Family` split, where both vocabularies have
the same term and still differ.

In [ ]:
confusion = pd.concat(confusions, ignore_index=True) if confusions else None
if confusion is None:
    print('no county cleared --min_confusion_rows; skipping the heatmap')
else:
    table = pd.crosstab(confusion['vote'], confusion['occupancy_type_mode'])
    # Keep the classes the delivery is actually about; a long tail of
    # one-row categories makes an unreadable grid.
    keep = table.sum(axis=1).sort_values(ascending=False).head(12).index
    table = table.loc[sorted(keep)]
    publish(table, 'permit-confusion')

    shares = table.div(table.sum(axis=1), axis=0)
    fig, ax = plt.subplots(
        figsize=(1.0 + 0.55 * table.shape[1], 1.4 + 0.42 * table.shape[0])
    )
    im = ax.imshow(
        shares.to_numpy(), cmap=MAGNITUDE_CMAP, vmin=0, vmax=1, aspect='auto'
    )
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=45, ha='right')
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.tick_params(length=0)
    for r in range(table.shape[0]):
        for c in range(table.shape[1]):
            n = table.iat[r, c]
            if n:
                # Ink flips on the dark end so the count stays legible.
                ax.text(
                    c,
                    r,
                    f'{n:,}',
                    ha='center',
                    va='center',
                    fontsize=7,
                    color='white' if shares.iat[r, c] > 0.55 else INK,
                )
    fig.colorbar(im, ax=ax, shrink=0.7, label='share of the delivered class')
    ax.set_xlabel('permit occupancy')
    ax.set_ylabel('delivered occupancy')
    ax.set_title(
        f'Delivered class vs permits, {confusion["county"].nunique()} counties',
        fontsize=11,
        pad=10,
    )
    publish(fig, 'permit-confusion')
    plt.show()

## Manifest

A short README so the folder explains itself to someone who receives the
bundle without this repository.

In [ ]:
lines = [
    f'# Accuracy reporting for {args.recipe_id}',
    '',
    'Scored against two references, neither of them an input to the inventory.',
    '',
    f'- CHEER hand-labeled survey points: {len(linked):,} points, '
    f'{linked["admin3_id_inv"].nunique() if "admin3_id_inv" in linked else len(counties)} counties.',
    f'- Shovels building permits: {len(permit_accuracy)} counties with scorable evidence.',
    '',
    'Permit coverage varies by county by two orders of magnitude, so read',
    'every agreement number next to the count beside it.',
    '',
    '## Files',
    '',
]
lines += [f'- `{name}`' for name in sorted(written)]
readme = out_dir / 'README.md'
readme.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(f'wrote {len(written)} files + README.md to {out_dir}')
for name in sorted(written):
    print(f'  {name}')

---
# Convert to script

*The above line and heading identify the end of the script.*

In [ ]:
from openplaces.flow import convert_to_script

try:
    convert_to_script(commit=True)
except Exception as error:
    # Headless execution (nbconvert) has no notebook context to resolve
    # the caller path from; run this cell interactively to commit the
    # script. Stripped from the converted script either way.
    print(f'convert_to_script skipped: {error}')

# Test script

In [ ]:
# from openplaces.flow import test_script

# test_script(*args_list, committed=True)

# Inspect results

In [ ]:
sorted(p.name for p in out_dir.iterdir())